## 1. State: 에이전트 메모리 시스템

In [1]:
from typing import TypedDict, List, Optional, Dict, Any, Annotated
from langgraph.graph import MessagesState
import operator

class AdvancedReActState(MessagesState):
    """ReAct 에이전트의 고급 상태 관리 스키마"""

    # 핵심 ReAct 컴포넌트
    thoughts: Annotated[List[str], operator.add]       # 사고 과정 기록
    actions: Annotated[List[Dict[str, Any]], operator.add]  # 수행한 행동들
    observations: Annotated[List[str], operator.add]   # 관찰 결과들

    # 메타데이터
    iteration_count: int                               # 현재 반복 횟수
    max_iterations: int                                # 최대 반복 제한

    # 실행 컨텍스트
    current_goal: str                                  # 현재 해결 중인 목표

    # 결과 관리
    final_answer: str                                  # 최종 답변
    is_complete: bool                                  # 완료 여부


In [2]:
def add_thought(state: AdvancedReActState, thought: str) -> dict:
    """새로운 사고를 상태에 추가"""
    return {
        "thoughts": [thought],          # Annotated[List, operator.add] 이므로 누적
        "iteration_count": state["iteration_count"] + 1,
    }

def record_action(state: AdvancedReActState, tool_name: str, args: dict) -> dict:
    """행동을 기록"""
    action = {
        "tool_name": tool_name,
        "arguments": args,
    }
    return {"actions": [action]}


## 2. Nodes: ReAct 단계별 처리 노드

### 2.1 Think 노드

In [3]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

# 도구 정의
@tool
def web_search(query: str) -> str:
    """웹에서 정보를 검색합니다."""
    # 실제 환경에서는 SerpAPI, Brave Search 등을 연동
    return f"'{query}' 검색 결과: 관련 정보를 찾았습니다."

@tool
def calculator(expression: str) -> str:
    """수학 표현식을 계산합니다."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {e}"

tools = [web_search, calculator]

# LLM 초기화 (LangChain 1.0 스타일)
llm = init_chat_model("openai:gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def think_node(state: AdvancedReActState) -> dict:
    """사고 노드: LLM이 현재 상태를 분석하고 다음 행동을 결정"""

    if state["iteration_count"] >= state.get("max_iterations", 10):
        return {
            "thoughts": ["최대 반복 횟수에 도달했습니다. 현재까지의 정보로 결론을 내리겠습니다."],
            "is_complete": True,
        }

    # 시스템 메시지 구성
    system_content = """당신은 체계적으로 문제를 해결하는 ReAct 에이전트입니다.
현재 상황을 분석하고 다음 행동을 신중히 결정하세요.

추론 가이드라인:
1. 현재 목표와 진행 상황을 명확히 파악
2. 이전 관찰 결과를 바탕으로 추론
3. 가장 효과적인 다음 단계 결정
"""

    messages = [SystemMessage(content=system_content)] + state["messages"]
    response = llm_with_tools.invoke(messages)

    return {
        "messages": [response],
        "thoughts": [response.content] if response.content else [],
    }


### 2.2 Act 노드 (ToolNode)

In [4]:
from langgraph.prebuilt import ToolNode

# LangGraph 내장 ToolNode 사용 (도구 실행 담당)
tool_node = ToolNode(tools)


### 2.3 Observe 노드

In [5]:
from langchain.messages import ToolMessage

def observe_node(state: AdvancedReActState) -> dict:
    """관찰 노드: 도구 실행 결과를 분석하고 관찰 기록 생성"""

    # 최신 ToolMessage 추출
    tool_messages = [
        msg for msg in state["messages"]
        if isinstance(msg, ToolMessage)
    ]

    if not tool_messages:
        return {}

    latest_result = tool_messages[-1]
    observation = f"도구 '{latest_result.name}' 결과: {latest_result.content}"

    return {"observations": [observation]}


## 3. Conditional Edges: 의사결정 라우팅

In [6]:
def should_continue(state: AdvancedReActState) -> str:
    """에이전트가 계속 실행할지, 종료할지 결정"""

    # 완료 상태 확인
    if state.get("is_complete", False):
        return "finish"

    # 최대 반복 횟수 초과
    if state["iteration_count"] >= state.get("max_iterations", 10):
        return "finish"

    # 마지막 메시지에 도구 호출이 있는지 확인
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "act"

    return "finish"


## 4. 전체 그래프 조립

In [ ]:
from langgraph.graph import StateGraph, END, START

def create_react_graph() -> StateGraph:
    """ReAct 에이전트 그래프 생성"""

    def initialize(state: AdvancedReActState) -> dict:
        """초기 상태 설정"""
        return {
            "iteration_count": 0,
            "max_iterations": 10,
            "current_goal": state["messages"][0].content if state["messages"] else "",
            "thoughts": [],
            "actions": [],
            "observations": [],
            "is_complete": False,
            "final_answer": "",
        }

    graph = StateGraph(AdvancedReActState)

    # 노드 등록
    graph.add_node("initialize", initialize)
    graph.add_node("think", think_node)
    graph.add_node("act", tool_node)
    graph.add_node("observe", observe_node)

    # 엣지 연결
    graph.add_edge(START, "initialize")
    graph.add_edge("initialize", "think")
    graph.add_conditional_edges(
        "think",
        should_continue,
        {
            "act": "act",
            "finish": END,
        }
    )
    graph.add_edge("act", "observe")
    graph.add_edge("observe", "think")

    return graph.compile()

# 그래프 생성 및 실행
react_agent = create_react_graph()

result = react_agent.invoke({
    "messages": [{"role": "user", "content": "파이썬 언어의 특징을 검색해서 알려주세요."}]
})

print(result["messages"][-1].content)
